# DÉCLIC Phase 6 — AMME
Les anciens résultats SingPAD sont invalides : le tri de dates est corrigé et
les sorties sont séparées dans phase6. Ne pas reprendre les checkpoints Phase 5.
Entraînement PyTorch réel. Ouvrir ce notebook, choisir **Exécution → Modifier le type
d'exécution → GPU** si disponible, puis **Tout exécuter**. Le CPU fonctionne aussi.
Autoriser Google Drive quand Google le demande. Aucun abonnement payant nécessaire.

La sauvegarde est faite toutes les 25 minibatches ou deux minutes, ainsi qu'à chaque
époque. Après déconnexion, relancer les mêmes cellules : le dernier checkpoint intact
est retrouvé automatiquement. Le travail depuis la dernière sauvegarde est refait.
Les checkpoints contiennent le modèle, l'optimiseur et l'état aléatoire.

Déposer une fois `2gvty-osfstorage-archive.zip` dans `Mon Drive/DECLIC/datasets/`.
Si absent, un sélecteur d'importation est proposé. Les données ne sont pas publiées
sur GitHub. Les sorties restent dans votre Drive. Le benchmark conserve ses propres
concepts et son instrument ; il ne valide pas automatiquement les prévisions du piano
personnel de DÉCLIC. [Guide](https://github.com/MoallaMelek/Declics-Knowledge-Tracing-for-Learner-Progress-and-Skill-Prediction./blob/phase6/scientific-validation/docs/PHASE5_COLAB.md)

In [ ]:
from pathlib import Path
import subprocess, sys, os, json, shutil
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
SOURCE = 'amme'
EPOCHS = 12
SEED = 42
BATCH_SIZE = 16
# Changing architecture/seed uses a separate output folder. Resuming preserves the split.
MODELS = ['enhanced_gru']
REPO_URL = 'https://github.com/MoallaMelek/Declics-Knowledge-Tracing-for-Learner-Progress-and-Skill-Prediction..git'
REF = 'phase6/scientific-validation'
REPO_DIR = Path('/content/declic-axis7')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REF, REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Existing checkout retained for reproducible resume.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
import torch
print('PyTorch', torch.__version__, '| GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('CPU disponible. Le GPU gratuit est facultatif, selon disponibilité Colab.')
print('Code revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
from google.colab import drive, files
if not Path('/content/drive/MyDrive').is_dir():
    try:
        drive.mount('/content/drive', timeout_ms=600000)
    except ValueError as error:
        if not str(error).startswith('mount failed'):
            raise
        print('Reprise du montage Drive après une erreur temporaire.')
        drive.mount('/content/drive', force_remount=True, timeout_ms=600000)

DRIVE_ROOT = Path('/content/drive/MyDrive/DECLIC')
DATA_DIR = DRIVE_ROOT / 'datasets'
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_NAME = '2gvty-osfstorage-archive.zip'
drive_archive = DATA_DIR / ARCHIVE_NAME
if not drive_archive.is_file():
    print('Importer uniquement', ARCHIVE_NAME, '(conservé dans Drive pour les prochains lancements).')
    uploaded = files.upload()
    if ARCHIVE_NAME not in uploaded:
        raise FileNotFoundError('Le nom du fichier doit être ' + ARCHIVE_NAME)
    shutil.copyfile(ARCHIVE_NAME, drive_archive)
    del uploaded
LOCAL_ARCHIVE = Path('/content') / ARCHIVE_NAME
shutil.copyfile(drive_archive, LOCAL_ARCHIVE)
RUN_ROOT = DRIVE_ROOT / 'phase6' / SOURCE / f'seed-{SEED}'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('Données prêtes. Checkpoints et métriques :', RUN_ROOT)

## Entraîner et reprendre
Les apprenants d'entraînement, validation et test sont séparés. Chaque prévision
utilise seulement les groupes chronologiquement antérieurs. Pour SingPAD, toutes
les notes du même enregistrement et les enregistrements simultanés sont cachés
ensemble. AMME prédit le score du prochain bloc ; Yousician le prochain jour actif.
Les scores continus ne sont pas transformés artificiellement en « correct/incorrect ».
La référence est simplement la moyenne d'entraînement, sans modèle classique entraîné.

In [ ]:
from axis7.cloud_training import train, atomic_json
reports = {}
for name in MODELS:
    folder = RUN_ROOT / name
    reports[name] = train(SOURCE, LOCAL_ARCHIVE, folder, model_name=name,
                          epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED)
    print(name, json.dumps(reports[name]['test'], ensure_ascii=False))
key = 'log_loss' if SOURCE == 'singpad' else 'mae'
# Select on validation only, never on test results.
selected = min(reports, key=lambda name: reports[name]['validation'][key])
atomic_json(RUN_ROOT / 'selection.json', {'selected': selected, 'criterion': 'validation_' + key})
print('Architecture sélectionnée sur validation :', selected)
print('Éligible au benchmark :', reports[selected]['validation_gate']['accepted'])

In [ ]:
# Small portable package: weights, exact config, actual metrics and history.
# No raw observations, prepared cache or optimizer checkpoints exported.
# Split provenance is counts/digests; learner IDs remain in private resume files.
import zipfile
EXPORT = RUN_ROOT / f'{SOURCE}-benchmark-bundle.zip'
folder = RUN_ROOT / selected
with zipfile.ZipFile(EXPORT, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ('model.pt', 'metrics.json', 'metrics.csv', 'history.json'):
        z.write(folder / name, name)
print('Export sauvegardé :', EXPORT)
print('Rapport :', folder / 'metrics.json')
# Optional download to this computer. Drive already holds the persistent copy.
# files.download(str(EXPORT))

## Retour dans DÉCLIC
Le fichier exporté peut être téléchargé puis importé avec
`python -m axis7.musical_registry import --bundle CHEMIN.zip`.
L'import vérifie le checksum, le découpage, les paramètres et le seuil de validation.
Un modèle refusé reste consultable en recherche, sans être utilisé pour des prévisions.
Les endpoints de recherche exposent uniquement le domaine et la cible réellement
évalués. Les auto-évaluations du carnet piano ne deviennent pas des labels de réussite.